In [1]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:
import pandas as pd

# Load the training and test datasets
train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)

train.head()

Training data shape: (8693, 14)
Test data shape: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [4]:
train.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

In [5]:
missing = train.isnull().sum()

print(missing)

missing_percentage = (train.isnull().sum() / len(train)) * 100

print(missing_percentage.sort_values(ascending=False))

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64
CryoSleep       2.496261
ShoppingMall    2.392730
VIP             2.335212
HomePlanet      2.312205
Name            2.300702
Cabin           2.289198
VRDeck          2.162660
Spa             2.105142
FoodCourt       2.105142
Destination     2.093639
RoomService     2.082135
Age             2.059128
PassengerId     0.000000
Transported     0.000000
dtype: float64


In [6]:
print(train['Transported'].value_counts())

print(train['Transported'].value_counts(normalize=True))

Transported
True     4378
False    4315
Name: count, dtype: int64
Transported
True     0.503624
False    0.496376
Name: proportion, dtype: float64


In [7]:
print(pd.crosstab(train['HomePlanet'], train['Transported'], normalize='index'))
print(pd.crosstab(train['CryoSleep'], train['Transported'], normalize='index'))
print(pd.crosstab(train['VIP'], train['Transported'], normalize='index'))

Transported     False     True 
HomePlanet                     
Earth        0.576054  0.423946
Europa       0.341154  0.658846
Mars         0.476976  0.523024
Transported     False     True 
CryoSleep                      
False        0.671079  0.328921
True         0.182417  0.817583
Transported     False     True 
VIP                            
False        0.493668  0.506332
True         0.618090  0.381910


In [8]:
spending_cols = [
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

train[spending_cols].describe()

,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,224.687617,458.077203,173.729169,311.138778,304.854791
std,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,47.000000,76.000000,27.000000,59.000000,46.000000
max,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [9]:
for col in spending_cols:
    print(f"\n{col}")
    print(train.groupby('Transported')[col].mean())


RoomService
Transported
False    389.266066
True      63.098021
Name: RoomService, dtype: float64

FoodCourt
Transported
False    382.615930
True     532.691984
Name: FoodCourt, dtype: float64

ShoppingMall
Transported
False    167.566217
True     179.829972
Name: ShoppingMall, dtype: float64

Spa
Transported
False    564.382666
True      61.675531
Name: Spa, dtype: float64

VRDeck
Transported
False    543.629822
True      69.148131
Name: VRDeck, dtype: float64


In [10]:
features = [
    'HomePlanet',
    'CryoSleep',
    'Destination',
    'Age',
    'VIP',
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

X = train[features]
y = train['Transported']

X_test = test[features]

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

categorical_features = [
    'HomePlanet',
    'CryoSleep',
    'Destination',
    'VIP'
]

numerical_features = [
    'Age',
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=2000))
])

In [13]:
baseline_scores = cross_val_score(
    baseline_model,
    X,
    y,
    cv=cv,
    scoring='accuracy'
)

print("Baseline CV scores:", baseline_scores)
print("Mean CV accuracy:", baseline_scores.mean())
print("Standard deviation:", baseline_scores.std())

Baseline CV scores: [0.78895917 0.77515814 0.80046003 0.78998849 0.77560414]
Mean CV accuracy: 0.7860339957027271
Standard deviation: 0.009585107572809889


In [14]:
baseline_model.fit(X, y)
baseline_predictions = baseline_model.predict(X_test)

In [15]:
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': baseline_predictions
})

submission.to_csv('/kaggle/working/submission_baseline.csv', index=False)

submission.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,False


In [16]:
# Experiment 2: Create TotalSpend feature

train_exp2 = train.copy()
test_exp2 = test.copy()

train_exp2['TotalSpend'] = train_exp2[spending_cols].fillna(0).sum(axis=1)
test_exp2['TotalSpend'] = test_exp2[spending_cols].fillna(0).sum(axis=1)

print("Average TotalSpend by Transported:")
print(train_exp2.groupby('Transported')['TotalSpend'].mean())

Average TotalSpend by Transported:
Transported
False    2004.149247
True      885.689127
Name: TotalSpend, dtype: float64


In [17]:
features_exp2 = features + ['TotalSpend']

X_exp2 = train_exp2[features_exp2]
y_exp2 = train_exp2['Transported']
X_test_exp2 = test_exp2[features_exp2]

In [18]:
categorical_features_exp2 = [
    'HomePlanet',
    'CryoSleep',
    'Destination',
    'VIP'
]

numerical_features_exp2 = [
    'Age',
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck',
    'TotalSpend'
]

numeric_transformer_exp2 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer_exp2 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_exp2 = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_exp2, numerical_features_exp2),
        ('cat', categorical_transformer_exp2, categorical_features_exp2)
    ]
)

In [19]:
model_exp2 = Pipeline(steps=[
    ('preprocessor', preprocessor_exp2),
    ('model', LogisticRegression(max_iter=2000))
])

In [20]:
exp2_scores = cross_val_score(
    model_exp2,
    X_exp2,
    y_exp2,
    cv=cv,
    scoring='accuracy'
)

print("Experiment 2 CV scores:", exp2_scores)
print("Experiment 2 Mean CV accuracy:", exp2_scores.mean())
print("Experiment 2 Standard deviation:", exp2_scores.std())

Experiment 2 CV scores: [0.78895917 0.77515814 0.80046003 0.78998849 0.77617952]
Experiment 2 Mean CV accuracy: 0.7861490705013463
Experiment 2 Standard deviation: 0.009461861843735946


In [21]:
model_exp2.fit(X_exp2, y_exp2)

exp2_predictions = model_exp2.predict(X_test_exp2)

In [22]:
submission_exp2 = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': exp2_predictions
})

submission_exp2.to_csv(
    '/kaggle/working/submission_exp2.csv',
    index=False
)

submission_exp2.to_csv(
    '/kaggle/working/submission.csv',
    index=False
)

submission_exp2.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,False


In [23]:
# Experiment 3: SpendCount

train_exp3 = train_exp2.copy()
test_exp3 = test_exp2.copy()

train_exp3['SpendCount'] = (
    train_exp3[spending_cols].fillna(0).gt(0).sum(axis=1)
)

test_exp3['SpendCount'] = (
    test_exp3[spending_cols].fillna(0).gt(0).sum(axis=1)
)

print("Average SpendCount by Transported:")
print(train_exp3.groupby('Transported')['SpendCount'].mean())

Average SpendCount by Transported:
Transported
False    2.464195
True     1.016446
Name: SpendCount, dtype: float64


In [24]:
features_exp3 = features + ['TotalSpend', 'SpendCount']

X_exp3 = train_exp3[features_exp3]
y_exp3 = train_exp3['Transported']
X_test_exp3 = test_exp3[features_exp3]

In [25]:
categorical_features_exp3 = [
    'HomePlanet',
    'CryoSleep',
    'Destination',
    'VIP'
]

numerical_features_exp3 = [
    'Age',
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck',
    'TotalSpend',
    'SpendCount'
]

numeric_transformer_exp3 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer_exp3 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_exp3 = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_exp3, numerical_features_exp3),
        ('cat', categorical_transformer_exp3, categorical_features_exp3)
    ]
)

In [26]:
model_exp3 = Pipeline(steps=[
    ('preprocessor', preprocessor_exp3),
    ('model', LogisticRegression(max_iter=2000))
])

In [27]:
exp3_scores = cross_val_score(
    model_exp3,
    X_exp3,
    y_exp3,
    cv=cv,
    scoring='accuracy'
)

print("Experiment 3 CV scores:", exp3_scores)
print("Experiment 3 Mean CV accuracy:", exp3_scores.mean())
print("Experiment 3 Standard deviation:", exp3_scores.std())

Experiment 3 CV scores: [0.7947096  0.77515814 0.79758482 0.79056387 0.77905639]
Experiment 3 Mean CV accuracy: 0.7874145624212956
Experiment 3 Standard deviation: 0.008793735523819375


In [28]:
model_exp3.fit(X_exp3, y_exp3)

exp3_predictions = model_exp3.predict(X_test_exp3)

In [29]:
submission_exp3 = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': exp3_predictions
})

submission_exp3.to_csv(
    '/kaggle/working/submission_exp3.csv',
    index=False
)

submission_exp3.to_csv(
    '/kaggle/working/submission.csv',
    index=False
)

submission_exp3.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [30]:
# Experiment 4: Cabin features

train_exp4 = train_exp3.copy()
test_exp4 = test_exp3.copy()

train_exp4[['Deck', 'CabinNum', 'Side']] = (
    train_exp4['Cabin']
    .str.split('/', expand=True)
)

test_exp4[['Deck', 'CabinNum', 'Side']] = (
    test_exp4['Cabin']
    .str.split('/', expand=True)
)

train_exp4['CabinNum'] = pd.to_numeric(
    train_exp4['CabinNum'],
    errors='coerce'
)

test_exp4['CabinNum'] = pd.to_numeric(
    test_exp4['CabinNum'],
    errors='coerce'
)

print("Transported rate by Deck:")
print(
    pd.crosstab(
        train_exp4['Deck'],
        train_exp4['Transported'],
        normalize='index'
    )
)

print("\nTransported rate by Side:")
print(
    pd.crosstab(
        train_exp4['Side'],
        train_exp4['Transported'],
        normalize='index'
    )
)

Transported rate by Deck:
Transported     False     True 
Deck                           
A            0.503906  0.496094
B            0.265725  0.734275
C            0.319946  0.680054
D            0.566946  0.433054
E            0.642694  0.357306
F            0.560129  0.439871
G            0.483783  0.516217
T            0.800000  0.200000

Transported rate by Side:
Transported     False     True 
Side                           
P            0.548740  0.451260
S            0.444963  0.555037


In [31]:
features_exp4 = features + [
    'TotalSpend',
    'SpendCount',
    'Deck',
    'CabinNum',
    'Side'
]

X_exp4 = train_exp4[features_exp4]
y_exp4 = train_exp4['Transported']
X_test_exp4 = test_exp4[features_exp4]

In [32]:
categorical_features_exp4 = [
    'HomePlanet',
    'CryoSleep',
    'Destination',
    'VIP',
    'Deck',
    'Side'
]

numerical_features_exp4 = [
    'Age',
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck',
    'TotalSpend',
    'SpendCount',
    'CabinNum'
]

numeric_transformer_exp4 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer_exp4 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_exp4 = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_exp4, numerical_features_exp4),
        ('cat', categorical_transformer_exp4, categorical_features_exp4)
    ]
)

In [33]:
model_exp4 = Pipeline(steps=[
    ('preprocessor', preprocessor_exp4),
    ('model', LogisticRegression(max_iter=2000))
])

In [34]:
exp4_scores = cross_val_score(
    model_exp4,
    X_exp4,
    y_exp4,
    cv=cv,
    scoring='accuracy'
)

print("Experiment 4 CV scores:", exp4_scores)
print("Experiment 4 Mean CV accuracy:", exp4_scores.mean())
print("Experiment 4 Standard deviation:", exp4_scores.std())

Experiment 4 CV scores: [0.79413456 0.78838413 0.80161012 0.80322209 0.79056387]
Experiment 4 Mean CV accuracy: 0.7955829541070586
Experiment 4 Standard deviation: 0.005895682705764376


In [35]:
model_exp4.fit(X_exp4, y_exp4)

exp4_predictions = model_exp4.predict(X_test_exp4)

In [36]:
submission_exp4 = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': exp4_predictions
})

submission_exp4.to_csv(
    '/kaggle/working/submission_exp4.csv',
    index=False
)

submission_exp4.to_csv(
    '/kaggle/working/submission.csv',
    index=False
)

submission_exp4.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [37]:
# Experiment 5: GroupSize

train_exp5 = train_exp4.copy()
test_exp5 = test_exp4.copy()

train_exp5['GroupId'] = train_exp5['PassengerId'].str.split('_').str[0]
test_exp5['GroupId'] = test_exp5['PassengerId'].str.split('_').str[0]

all_group_ids = pd.concat([
    train_exp5['GroupId'],
    test_exp5['GroupId']
])

group_sizes = all_group_ids.value_counts()

train_exp5['GroupSize'] = train_exp5['GroupId'].map(group_sizes)
test_exp5['GroupSize'] = test_exp5['GroupId'].map(group_sizes)

print("Missing GroupSize in training:", train_exp5['GroupSize'].isnull().sum())
print("Missing GroupSize in test:", test_exp5['GroupSize'].isnull().sum())

print("\nAverage GroupSize by Transported:")
print(train_exp5.groupby('Transported')['GroupSize'].mean())



Missing GroupSize in training: 0
Missing GroupSize in test: 0

Average GroupSize by Transported:
Transported
False    1.902665
True     2.166514
Name: GroupSize, dtype: float64


In [38]:
features_exp5 = features + [
    'TotalSpend',
    'SpendCount',
    'Deck',
    'CabinNum',
    'Side',
    'GroupSize'
]

X_exp5 = train_exp5[features_exp5]
y_exp5 = train_exp5['Transported']
X_test_exp5 = test_exp5[features_exp5]

In [39]:

categorical_features_exp5 = [
    'HomePlanet',
    'CryoSleep',
    'Destination',
    'VIP',
    'Deck',
    'Side'
]

numerical_features_exp5 = [
    'Age',
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck',
    'TotalSpend',
    'SpendCount',
    'CabinNum',
    'GroupSize'
]

numeric_transformer_exp5 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer_exp5 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_exp5 = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_exp5, numerical_features_exp5),
        ('cat', categorical_transformer_exp5, categorical_features_exp5)
    ]
)

In [40]:
model_exp5 = Pipeline(steps=[
    ('preprocessor', preprocessor_exp5),
    ('model', LogisticRegression(max_iter=2000))
])

In [41]:
exp5_scores_fixed = cross_val_score(
    model_exp5,
    X_exp5,
    y_exp5,
    cv=cv,
    scoring='accuracy'
)

print("Experiment 5 CV scores:", exp5_scores_fixed)
print("Experiment 5 Mean CV accuracy:", exp5_scores_fixed.mean())
print("Experiment 5 Standard deviation:", exp5_scores_fixed.std())

Experiment 5 CV scores: [0.79815986 0.78838413 0.79988499 0.80379747 0.78941312]
Experiment 5 Mean CV accuracy: 0.7959279138110271
Experiment 5 Standard deviation: 0.006031934912860044


In [42]:
model_exp5.fit(X_exp5, y_exp5)

exp5_predictions = model_exp5.predict(X_test_exp5)

In [43]:
submission_exp5 = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': exp5_predictions
})

submission_exp5.to_csv(
    '/kaggle/working/submission_exp5.csv',
    index=False
)

submission_exp5.to_csv(
    '/kaggle/working/submission.csv',
    index=False
)

submission_exp5.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [44]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

In [45]:
logistic_model = Pipeline(steps=[
    ('preprocessor', preprocessor_exp5),
    ('model', LogisticRegression(max_iter=2000))
])

logistic_scores = cross_val_score(
    logistic_model,
    X_exp5,
    y_exp5,
    cv=cv,
    scoring='accuracy'
)

print("Logistic Regression")
print("Mean:", logistic_scores.mean())
print("Std:", logistic_scores.std())

Logistic Regression
Mean: 0.7959279138110271
Std: 0.006031934912860044


In [46]:
random_forest_model = Pipeline(steps=[
    ('preprocessor', preprocessor_exp5),
    ('model', RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

rf_scores = cross_val_score(
    random_forest_model,
    X_exp5,
    y_exp5,
    cv=cv,
    scoring='accuracy'
)

print("Random Forest")
print("Mean:", rf_scores.mean())
print("Std:", rf_scores.std())

Random Forest
Mean: 0.8016784774393176
Std: 0.01059784879956792


In [47]:
gradient_model = Pipeline(steps=[
    ('preprocessor', preprocessor_exp5),
    ('model', GradientBoostingClassifier(
        random_state=42
    ))
])

gradient_scores = cross_val_score(
    gradient_model,
    X_exp5,
    y_exp5,
    cv=cv,
    scoring='accuracy'
)

print("Gradient Boosting")
print("Mean:", gradient_scores.mean())
print("Std:", gradient_scores.std())

Gradient Boosting
Mean: 0.805590557381562
Std: 0.005904236508057436


In [48]:
model_comparison = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Random Forest',
        'Gradient Boosting'
    ],
    'Mean CV Accuracy': [
        logistic_scores.mean(),
        rf_scores.mean(),
        gradient_scores.mean()
    ],
    'Standard Deviation': [
        logistic_scores.std(),
        rf_scores.std(),
        gradient_scores.std()
    ]
})

model_comparison

,Model,Mean CV Accuracy,Standard Deviation
0,Logistic Regression,0.795928,0.006032
1,Random Forest,0.801678,0.010598
2,Gradient Boosting,0.805591,0.005904


In [49]:
from sklearn.model_selection import GridSearchCV

gradient_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_exp5),
    ('model', GradientBoostingClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [2, 3]
}

grid_search = GridSearchCV(
    gradient_pipeline,
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_exp5, y_exp5)

print("Best parameters:")
print(grid_search.best_params_)

print("Best CV accuracy:")
print(grid_search.best_score_)

Best parameters:
{'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}
Best CV accuracy:
0.805935715604447


In [50]:
best_model = grid_search.best_estimator_

best_model.fit(X_exp5, y_exp5)

final_predictions = best_model.predict(X_test_exp5)

final_submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': final_predictions
})

final_submission.to_csv('/kaggle/working/submission_final.csv', index=False)
final_submission.to_csv('/kaggle/working/submission.csv', index=False)

final_submission.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


# Iteration Log

| # | What I changed | Why I expected it to help | CV score | Leaderboard | What I concluded |
|---|---|---|---|---|---|
| 1 | Built a Logistic Regression baseline using passenger, travel, age, and spending features with imputation, one-hot encoding, and scaling. | I needed a reliable starting point before adding new features. | 78.60% ± 0.96% | 79.050% | The baseline provided a good reference for later experiments. |
| 2 | Added `TotalSpend`, the total amount spent across the five spending categories. | A passenger's overall spending might contain useful information about whether they were transported. | 78.61% ± 0.95% | 79.050% | The improvement was almost zero, so `TotalSpend` was not very useful. |
| 3 | Added `SpendCount`, the number of spending categories where the passenger spent money. | I expected the number of different services used to capture passenger behavior better than total spending alone. | 78.74% ± 0.88% | 79.494% | This gave a small improvement, so the feature was useful enough to keep. |
| 4 | Added Cabin information: `Deck`, `CabinNum`, and `Side`. | Cabin location could be related to where passengers were located on the ship and their transportation outcome. | 79.56% ± 0.59% | 79.658% | This produced a clear improvement and became an important feature-engineering step. |
| 5 | Added `GroupSize` based on the passenger's travel group. | Passengers travelling in groups might have different transportation patterns from passengers travelling alone. | 79.59% ± 0.60% | 79.541% | CV improved slightly, although the leaderboard score decreased slightly. |
| 6 | Compared Logistic Regression, Random Forest, and Gradient Boosting using the same features and CV folds. | I wanted to determine which model family performed best before tuning. | 80.56% ± 0.59% | — | Gradient Boosting had the highest mean CV accuracy, so I selected it for tuning. |
| 7 | Tuned Gradient Boosting using GridSearchCV. | I expected better hyperparameters to improve the model's performance. | 80.59% | 79.845% | Tuning slightly improved CV and produced our best Kaggle score. |

## Final Result

- **Best CV Accuracy:** 80.59%
- **Best Kaggle Score:** 79.845%
- **Best Kaggle Version:** V17